# Customer Churn Prediction & Advanced AI Analytics\n\n**Educational demonstration using synthetic data only.** Results are not validated real-world business performance.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score

## 1. Generate synthetic data

In [ ]:
def make_synthetic_data(n=1200, random_state=42):
    rng = np.random.default_rng(random_state)
    tenure = rng.integers(0, 73, n)
    monthly_charges = np.round(rng.normal(70, 22, n).clip(18, 130), 2)
    contract = rng.choice(["Month-to-month", "One year", "Two year"], n, p=[0.55, 0.25, 0.20])
    support_calls = rng.poisson(1.5, n).clip(0, 8)
    internet_service = rng.choice(["Fiber", "DSL", "None"], n, p=[0.5, 0.35, 0.15])
    logit = (1.0 - 0.055*tenure + 0.018*(monthly_charges-70)
             + 0.50*(contract=="Month-to-month") + 0.28*support_calls
             + 0.25*(internet_service=="Fiber"))
    probability = 1/(1+np.exp(-logit))
    churn = rng.binomial(1, probability)
    return pd.DataFrame({"tenure_months":tenure, "monthly_charges":monthly_charges,
                         "contract":contract, "support_calls":support_calls,
                         "internet_service":internet_service, "churn":churn})
df = make_synthetic_data()
df.head()

## 2. Explore the data

In [ ]:
print("Shape:", df.shape)
display(df.describe(include="all"))
display(df["churn"].value_counts(normalize=True).rename("proportion"))

## 3. Train Logistic Regression

In [ ]:
X, y = df.drop(columns="churn"), df["churn"]
numeric = ["tenure_months", "monthly_charges", "support_calls"]
categorical = ["contract", "internet_service"]
preprocess = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical)
])
model = Pipeline([("preprocess", preprocess), ("classifier", LogisticRegression(max_iter=1000, random_state=42))])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model.fit(X_train, y_train)
pred = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]
print("Model trained.")

## 4. Evaluate the model

In [ ]:
print(f"Accuracy: {accuracy_score(y_test, pred):.3f}")
try:
    print(f"ROC-AUC: {roc_auc_score(y_test, probabilities):.3f}")
except ValueError:
    print("ROC-AUC unavailable")
print(classification_report(y_test, pred, zero_division=0))

In [ ]:
ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=["Stay", "Churn"]).plot()
plt.title("Confusion Matrix — Synthetic Demo Data")
plt.tight_layout()
plt.show()

## 5. Save data and note limitations

In [ ]:
out = Path("outputs")
out.mkdir(exist_ok=True)
df.to_csv(out / "synthetic_customer_churn_data.csv", index=False)
print("Saved synthetic data to", out.resolve())

This project uses generated synthetic data only. Metrics are for learning and are not validated on real customers. Real-world use requires authorized data, privacy safeguards, careful validation, bias/leakage checks, and monitoring.